# Uploading images to S3

In [1]:
import boto3
s3 = boto3.client('s3')
response = s3.list_buckets()
print([bucket['Name'] for bucket in response['Buckets']])

['macrohet.glimpses']


In [2]:
import boto3
import os
from pathlib import Path
from tqdm.notebook import tqdm
import time

class S3ImageUploader:
    def __init__(self, bucket_name: str, base_folder: str, s3_prefix: str = ""):
        self.s3_client = boto3.client('s3')
        self.bucket_name = bucket_name
        self.base_folder = Path(base_folder)
        self.s3_prefix = f"{s3_prefix.rstrip('/')}/" if s3_prefix else ""
        
    def count_files_in_folder(self, folder: Path) -> int:
        """Count number of tif files in a folder"""
        return len(list(folder.glob('*.png*')))
    
    def verify_file_exists(self, s3_key: str) -> bool:
        """Verify if a file exists in S3"""
        try:
            self.s3_client.head_object(Bucket=self.bucket_name, Key=s3_key)
            return True
        except:
            return False
            
    def verify_existing_uploads(self):
        """Verify what's already in the specified S3 directory."""
        print(f"\nChecking existing files in {self.bucket_name}/{self.s3_prefix}...")
        existing_files = {}
        
        try:
            paginator = self.s3_client.get_paginator('list_objects_v2')
            pages = paginator.paginate(Bucket=self.bucket_name, Prefix=self.s3_prefix)
            
            # First count total pages
            total_pages = sum(1 for _ in pages)
            
            # Reset pagination for actual processing
            pages = paginator.paginate(Bucket=self.bucket_name, Prefix=self.s3_prefix)
            
            with tqdm(total=total_pages, desc="Checking S3", unit="page") as pbar:
                for page in pages:
                    if 'Contents' in page:
                        for obj in page['Contents']:
                            relative_key = obj['Key'][len(self.s3_prefix):]
                            path_parts = relative_key.split('/')
                            if len(path_parts) > 1:
                                folder = path_parts[0]
                                filename = path_parts[-1]
                                if folder not in existing_files:
                                    existing_files[folder] = set()
                                existing_files[folder].add(filename)
                    pbar.update(1)
            
            print(f"Found {len(existing_files)} folders with existing uploads")
            return existing_files
        except Exception as e:
            print(f"Error checking existing files: {e}")
            return {}
            
    
    def upload_file(self, local_path: Path, s3_key: str, pbar=None) -> bool:
        """Upload a single file to S3 with verification"""
        try:
            # Try to upload file
            self.s3_client.upload_file(
                str(local_path),
                self.bucket_name,
                s3_key,
                ExtraArgs={'ContentType': 'image/png'}
            )
            
            # Verify upload was successful
            if not self.verify_file_exists(s3_key):
                print(f"\nWarning: Upload verification failed for {s3_key}")
                return False
                
            if pbar:
                pbar.update(1)
            return True
            
        except Exception as e:
            print(f"\nError uploading {s3_key}: {str(e)}")
            return False

    def process_folder(self, folder: Path, existing_files: dict):
        """Process a single folder of images with detailed progress tracking"""
        folder_name = folder.name
        
        # Count files that need uploading
        local_files = set(f.name for f in folder.glob('*.png*'))
        existing_folder_files = existing_files.get(folder_name, set())
        files_to_upload = local_files - existing_folder_files
        
        if not files_to_upload:
            print(f"Skipping {folder_name} - all {len(local_files)} files already uploaded")
            return True
            
        print(f"\nProcessing {folder_name} - {len(files_to_upload)} files to upload")
        
        success = True
        with tqdm(total=len(files_to_upload), 
                 desc=f"Files in {folder_name}", 
                 unit="file") as pbar:
            for file_name in files_to_upload:
                local_path = folder / file_name
                s3_key = f"{self.s3_prefix}{folder_name}/{file_name}"
                
                if not self.upload_file(local_path, s3_key, pbar):
                    success = False
                    print(f"Failed to upload {file_name}")
                
                # Small delay to prevent throttling
                time.sleep(0.1)
        
        return success

    def run_upload(self):
        """Main upload process with detailed progress tracking"""
        print("\nStarting upload process...")
        
        # Verify existing uploads first
        existing_files = self.verify_existing_uploads()
        
        # Get list of folders and count total files
        folders = [f for f in sorted(self.base_folder.glob('*/')) if f.is_dir()]
        total_folders = len(folders)
        total_files = sum(self.count_files_in_folder(f) for f in folders)
        
        print(f"\nFound {total_folders} folders containing {total_files} files")
        
        # Process folders with progress tracking
        with tqdm(total=total_folders, 
                 desc="Overall Progress", 
                 unit="folder") as folder_pbar:
            
            for folder in folders:
                success = self.process_folder(folder, existing_files)
                
                if success:
                    print(f"✓ Completed folder: {folder.name}")
                else:
                    print(f"⚠ Some files failed in folder: {folder.name}")
                
                folder_pbar.update(1)


In [4]:
uploader = S3ImageUploader(
    bucket_name="macrohet.glimpses",
    base_folder="/mnt/SYNO/macrohet_syno/results/glimpse_store/glimpse_frames",
    s3_prefix="glimpse_frames"
)

# Optional: verify existing uploads first
existing = uploader.verify_existing_uploads()

# Run the upload with progress bars
uploader.run_upload()


Checking existing files in macrohet.glimpses/glimpse_frames/...


Checking S3:   0%|          | 0/70 [00:00<?, ?page/s]

Found 1111 folders with existing uploads

Starting upload process...

Checking existing files in macrohet.glimpses/glimpse_frames/...


Checking S3:   0%|          | 0/70 [00:00<?, ?page/s]

Found 1111 folders with existing uploads

Found 1154 folders containing 72285 files


...

✓ Completed folder: 97.3.4.ND0003
Skipping 974.4.5.PS0000 - all 72 files already uploaded
✓ Completed folder: 974.4.5.PS0000
Skipping 978.3.4.ND0003 - all 46 files already uploaded
✓ Completed folder: 978.3.4.ND0003
Skipping 979.3.4.ND0003 - all 44 files already uploaded
✓ Completed folder: 979.3.4.ND0003

Processing 98.3.4.ND0003 - 78 files to upload


Files in 98.3.4.ND0003:   0%|          | 0/78 [00:00<?, ?file/s]

✓ Completed folder: 98.3.4.ND0003
Skipping 980.3.4.ND0003 - all 71 files already uploaded
✓ Completed folder: 980.3.4.ND0003
Skipping 986.3.4.ND0003 - all 71 files already uploaded
✓ Completed folder: 986.3.4.ND0003
Skipping 986.4.4.ND0002 - all 44 files already uploaded
✓ Completed folder: 986.4.4.ND0002

Processing 99.3.4.ND0003 - 78 files to upload


Files in 99.3.4.ND0003:   0%|          | 0/78 [00:00<?, ?file/s]

✓ Completed folder: 99.3.4.ND0003
Skipping 991.4.5.PS0000 - all 71 files already uploaded
✓ Completed folder: 991.4.5.PS0000
Skipping 996.4.5.PS0000 - all 71 files already uploaded
✓ Completed folder: 996.4.5.PS0000
